## Type I - III diffusable nodes based on the Diego et al. 2018 using and Random matrix theory (RMT)

**Diffusion Rates:**

**Type I**   
DU, DV, DW = 1.0, 0.0, 10.0   (v immobile → this gives much higher values so lets do 1.0, 0, 1.0 ok so apparently we cant do that it has to be higher!!!!)  

**Type II**  
DU, DV, DW = 1.0, 1.0, 0.0    (w immobile)  

**Type III**  
DU, DV, DW = 0.0, 1.0, 1.0    (u immobile)  

Then for the LHS/robustness comparison, scan d from 0.1 to 10 and measure what fraction of our stable samples still gives Turing instability at each d value.

We should see Type I collapse to zero as d → 1, while Type II and III stay robustly non-zero...

In [3]:
# Adjacency Matrix

#       u  v  w
#     ┌─────────┐
#  u  │ 0  1  0 │  
#  v  │ 1  0  1 │
#  w  │ 1  1  1 │
#     └─────────┘

# The destabilizing module is the u-v mutual activation cycle: fuv * fvu > 0, this is the positive feedback loop that drives instability.

In [4]:
import numpy as np
from numpy.linalg import eigvals
from scipy.optimize import fsolve
import matplotlib.pyplot as plt
from scipy.linalg import eig

### Type I

_v is immobile, w and u > 1_

In [5]:
adjacency_matrix = np.array([
    [0, 1, 0],
    [1, 0, 1],
    [1, 1, 0],
])

# the diagonal of the adjacency should always be 0
# the w self-loop [2,2] is already handled by J = G - I giving diagonal = -1.
# self-decay handled by J = G - I
# adjacency_matrix[i,j] = 1 means species j affects species i

In [12]:

# apply sign constraints for specific topology and type (activating (+) or inhibiting (-))
# for type I, the u-v cycle is destabilising and the v-w cycle is stabilising, so we have:

def sign_constraints(J):
    J[0, 1] =  abs(J[0, 1]) # v activates u, u-v destabilising
    J[1, 0] =  abs(J[1, 0]) # u activates v
    J[1, 2] =  abs(J[1, 2]) # w activates v
    J[2, 1] = -abs(J[2, 1]) # v inhibits w, v-w cycle must be stabilising → edges opposite sign
    # J[2, 0] unconstrained

    return J

# template:
# J[i, j] =  abs(J[i, j])   # j activates i
# J[i, j] = -abs(J[i, j])   # j inhibits i
# leave unconstrained edges untouched

# so we don't do: J = G - np.eye(3), because it has no sign constraints, and we want to enforce the sign constraints for the specific topology and type

In [13]:
def generate_jacobian_type1(sigma):
    
    # random matrix
    G = np.random.normal(0, sigma, (3, 3))
    np.fill_diagonal(G, 0)
    
    # diagonal becomes -1, self-decay handled by J = G - I, off-diagonal from N(0, sigma)
    J = G - np.eye(3)
    
    # apply sparsity mask after sampling from adjacency matrix, but only to off-diagonal elements, the diagonal is already -1 from J = G - I
    for i in range(3): 
        for j in range(3):
            if i != j and adjacency_matrix[i, j] == 0:
                J[i, j] = 0

    J = sign_constraints(J)

    return J

# J = G - I  (RMT convention, May 1972)

In [ ]:
def is_stable(J):
    return np.all(np.real(eigvals(J)) < 0)

In [ ]:
# check turing instabilitiy: does diffusion destabilise a mode that was stable without diffusion?

# two ways to check turing stability:
# 1. shaberi et al (2025)
#    - compute all eigenvalues and check if real part < 0
#    - detects any instability, including oscillatory ones (complex eigenvalues)
#    - detects Turing I only (defined wavelength, restabilises at large k)
# 2. diego et al (2018) 
#    - characteristic polynomial and Routh-Hurwitz criteria
#    - detects only stationary instabilities, not oscillary ones
#    - condition: a3 < 0 AND a1 > 0 AND a2 > 0 for some k > 0.

def is_turing_shaberi(J, DU, DV, DW):
    D = np.diag([DU, DV, DW])
    any_unstable = False
    for k in np.arange(0.01, 10, 0.01):
        J_k = J - k**2 * D
        if np.max(np.real(eigvals(J_k))) > 0:
            any_unstable = True
            break
    
    if not any_unstable:
        return False
    
    J_large = J - 10**2 * D
    return np.all(np.real(eigvals(J_large)) < 0)



def is_turing_diego(J, DU, DV, DW):

    return idk

In [ ]:
# type I specifications

n_samples = 100
sigma = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]
DU, DV, DW = 1.0, 0.0, 10.0